In [70]:
%%capture
!apt-get update -qq && apt-get install -y -qq zstd && sudo apt install pciutils
!pip install langchain langchain-ollama langchain-text-splitters langchain-community deepagents pypdf langchain-core pycodestyle pycodestyle_magic flake8 nest_asyncio langchain-huggingface langchain-postgres udocker psycopg fastapi nest-asyncio pyngrok uvicorn
!udocker --allow-root install

%load_ext pycodestyle_magic

!curl -fsSL https://ollama.com/install.sh | sh

In [60]:
import subprocess
import time
import os

env = os.environ.copy()
subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=env
)

time.sleep(5)

!ollama run qwen2.5:7b > /dev/null 2>&1
!ollama ps

NAME          ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
qwen2.5:7b    845dbda0ea48    4.7 GB    100% GPU     4096       4 minutes from now    


In [2]:
import os
import httpx
import asyncio
import nest_asyncio

nest_asyncio.apply()

urls = {'q1':'https://s1.q4cdn.com/050606653/files/doc_financials/2026/q1/Q1-2026-Earnings-Release_vF.pdf', 'q2': 'https://s1.q4cdn.com/050606653/files/doc_financials/2026/q3/Q3-2026-Earnings-Release_vF.pdf', 'q3':'https://s1.q4cdn.com/050606653/files/doc_financials/2026/q2/Q2-2026-Earnings-Release_vF.pdf'}

async def fetch(url:str, quarter:str):
    async with httpx.AsyncClient() as client:
        r = await client.get(url)
        with open(f"visa_{quarter}.pdf", "wb") as f:
            f.write(r.content)

async def download_reports():
    async with httpx.AsyncClient() as client:
        tasks = [fetch(url, quarter) for quarter, url in urls.items()]
        await asyncio.gather(*tasks)

if not all(file in os.listdir(os.curdir) for file in ('visa_q3.pdf', 'visa_q1.pdf', 'visa_q2.pdf')):
    asyncio.run(download_reports())

os.listdir(os.curdir)

['.config',
 'postgres_log.txt',
 'visa_q2.pdf',
 'visa_q1.pdf',
 'visa_q3.pdf',
 'sample_data']

In [62]:
# !udocker --allow-root rm -f pgvector-container

# !udocker --allow-root run -d \
#   --name="pgvector-container" \
#   --userenv="POSTGRES_PASSWORD=postgres" \
#   --userenv="POSTGRES_USER=postgres" \
#   --userenv="POSTGRES_DB=financial_rag" \
#   "docker.io/pgvector/pgvector:pg16" > /dev/null 2>&1

In [63]:
# !udocker --allow-root ps

CONTAINER ID                         P M NAMES              IMAGE               
1b8c3fca-0f50-3f09-b9cd-4812306c3235 . W ['pgvector-container'] docker.io/pgvector/pgvector:pg16
60e078f8-f622-3553-9acb-845fcfc5b74c . W                    docker.io/pgvector/pgvector:pg16
2062232e-05b9-3211-8415-d393c23bec38 . W                    docker.io/pgvector/pgvector:pg16
f3683fe4-a6e7-3ca0-8b6b-42f6859db083 . W                    docker.io/pgvector/pgvector:pg16


In [64]:
# !udocker --allow-root run pgvector-container psql -U postgres -d financial_rag -c "DROP TABLE IF EXISTS financial_reports;"

In [69]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from pprint import pprint
from typing import Optional

from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain.messages import HumanMessage
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.tools import tool
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_postgres import PGVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field
from langchain_postgres import PGEngine
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")
vector_store = InMemoryVectorStore(embedding=embeddings)

class FinancialMetrics(BaseModel):
    company: str = Field(description="Company name, e.g., Visa")
    metric_name: str = Field(
        description="Name of the financial metric (e.g., Net Revenue)"
    )
    period: str = Field(description="Fiscal period or quarter (e.g., Q2)")
    value: float = Field(description="Exact numerical value without symbols or text")
    unit: str = Field(description="Unit of measurement (e.g., billions USD)")

class AgentResponse(BaseModel):
    found: bool = Field(description="True if the specific company and its metrics were found in the context. False otherwise.")
    metrics: Optional[FinancialMetrics] = Field(default=None, description="The financial metrics if found, otherwise None/null.")

def _process_single_pdf(file_path: Path) -> list[Document]:
    try:
        loader = PyPDFLoader(str(file_path))
        documents = loader.load()
        return RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200,
        ).split_documents(documents)
    except Exception as e:
        print(f"Failed to process {file_path.name}: {e}")
        return []


def split_pdf_content(files: list[Path], max_workers: int) -> list[Document]:
    all_documents: list[Document] = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_file = {
            executor.submit(_process_single_pdf, file_path): file_path
            for file_path in files
        }

        for future in as_completed(future_to_file):
            docs = future.result()
            all_documents.extend(docs)

    return all_documents


def load_embeddings(documents: list[Document]):
    vector_store.add_documents(documents=documents)

@tool
def search_pdf(query: str) -> str:
    """Search in vectore store about financial report from query

    Args:
        query (str): search query in document

    Returns:
        str: information about the data in the document
    """
    retriev: list[Document] = vector_store.similarity_search(query=query, k=4)
    return "\n\n".join([document.page_content for document in retriev])

dir: Path = Path.cwd().resolve()
max_workers: int = os.cpu_count() or 1

pdf_files: list[Path] = [file for file in dir.glob("*.pdf") if file.is_file()]

documents: list[Document] = split_pdf_content(pdf_files, max_workers)
load_embeddings(documents)

system_prompt = """
You are an expert financial research analyst.
Extract the requested metrics precisely based ONLY on the retrieved documents provided by your search_pdf tool.

CRITICAL RULES FOR STRUCTURED OUTPUT:
1. Search thoroughly through the context. Note that company names might have suffixes or variations (e.g., 'Visa Inc.', 'Visa', 'VISA' or 'NVIDIA Corporation'). Treat them as matching.
2. If the company or the specific metrics are present anywhere in the context, you MUST set `found=True` and extract the data accurately into the `metrics` object.
3. Only if there is absolutely no mention of the company or its financial data in the retrieved text, set `found=False` and `metrics=None`. Do not make up numbers.
"""

agent = create_agent(
    "ollama:qwen2.5:7b",
    tools=[search_pdf],
    system_prompt=system_prompt,
    response_format=AgentResponse,
)

company = "Visa"
EXAMPLE_QUERY = (
    f"Give me the Net Revenue from {company} in billions of dollars for Q1."
)

result = agent.invoke({"messages": [HumanMessage(content=EXAMPLE_QUERY)]})

structured_data = result.get("structured_response")

print(flush=True)

if not structured_data or not structured_data.found:
    print(f"Can't find data about {company} (structured_response=None)")
else:
    pprint(structured_data.metrics.model_dump())

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


{'company': 'Visa',
 'metric_name': 'Net Revenue',
 'period': 'Q1 2026',
 'unit': 'billion dollars',
 'value': 10.9}


In [3]:
import threading
import time
from fastapi import FastAPI
from uvicorn import Config, Server

app = FastAPI()

@app.get("/")
def root():
    return {"message": "Hello World"}

def start_uvicorn():
    config = Config(app, host="127.0.0.1", port=8000, log_level="info")
    server = Server(config)
    server.run()

thread = threading.Thread(target=start_uvicorn)
thread.start()

In [1]:
# !fuser -k 8000/tcp

In [4]:
!curl http://127.0.0.1:8000/

INFO:     127.0.0.1:50240 - "GET / HTTP/1.1" 200 OK
{"message":"Hello World"}